In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, bindparam
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import os
import urllib3

# Load environment variables from .env file
load_dotenv()

strPresto = ('presto://{username}:{password}@{ipaddress}:{port}/{dbname}/{schema}'
             .format(username=os.getenv('HIVE_SVC_USER'),
                     password=os.getenv('HIVE_SVC_PASS'),
                     ipaddress=os.getenv('HIVE_SVC_ADDRESS'),
                     port=os.getenv('HIVE_SVC_PORT'),
                     dbname=os.getenv('HIVE_SVC_DBNAME'),
                     schema=os.getenv('HIVE_SVC_SCHEMA')))
 
presto_engine = create_engine(strPresto, connect_args={"protocol": "https", "requests_kwargs": {"verify": False}})

# disable certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# ------------------------------------------------------------------
# Start date and end dates for dateselectors in queries
# ------------------------------------------------------------------

start_date = '2025-12-01' 
end_date = '2026-01-26' 

In [3]:
# ------------------------------------------------------------------
# LOB and Client list as found in the Smart Offer query (lowercase)
# ------------------------------------------------------------------

client_list = [
    "V",
    "A",
]

business_unit_list = [
    "S",
    "M",
]


In [4]:
# ------------------------------------------------------------------
# Load SQL template for metric query
# ------------------------------------------------------------------

sql_path = "SQL Queries/expert_assist_data_week.sql"  # <- make sure this path is correct

with open(sql_path, "r") as f:
    EXPERT_ASSIST_SQL_TEMPLATE = f.read()

print("Loaded SQL template:")
print(EXPERT_ASSIST_SQL_TEMPLATE[:500], "...")

Loaded SQL template:
WITH
DateSelector AS (
    SELECT *
    FROM
        ( VALUES (
             DATE (:start_date), DATE (:end_date),
                 :start_date, :end_date
                 ))
        AS t ("StartDate","EndDate","StartDateStr","EndDateStr")
),

Expert_Assist_Per_Call AS(
    SELECT 
        COALESCE(
            element_at(ex.edp_raw_data_map, 'Identities_ReservationSid'),
            element_at(ex.edp_raw_data_map, 'Identities_SessionId')
            ) as reservation_id,
        COUNT(DISTINCT e ...


In [5]:
### Compile SQL With Literal Binds (Code)

#This uses your proven pattern: bind params + `literal_binds=True`.

#python
# ------------------------------------------------------------------
# Build literal SQL for Presto using SQLAlchemy binds
# This allows us to use expanding=True for metric_list and still
# send flattened literal SQL to Presto.
# ------------------------------------------------------------------

def compile_presto_sql(
    sql_template: str,
    engine,
    start_date,
    end_date,
    client_list,
    business_unit_list,
):
    """
    Creates literal SQL for Presto by binding parameters and compiling
    with literal_binds=True.
    """
    
    stmt = text(sql_template).bindparams(
        bindparam("start_date", value=start_date),
        bindparam("end_date", value=end_date),
        bindparam("client_list", value=list(client_list), expanding=True),
        bindparam("business_unit_list", value=list(business_unit_list), expanding=True),
    )

    compiled = stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )

    return str(compiled)

In [6]:
# ------------------------------------------------------------------
# Query Presto for metrics, client groups, and date range, returning the
# aggregated metrics DataFrame.
# ------------------------------------------------------------------

def query_expert_assist_presto_group(
    start_date,
    end_date,
    client_list,
    business_unit_list,
):
    """
    Execute the Presto query for a list of experts over a date range.

    Returns DataFrame with:
      [expert_id, metric, icp_client, site, num, den, calc]
    """

    sql = compile_presto_sql(
        sql_template=EXPERT_ASSIST_SQL_TEMPLATE,
        engine=presto_engine,
        start_date=start_date,
        end_date=end_date,
        client_list=client_list,
        business_unit_list=business_unit_list,
    )

    # Uncomment to debug generated SQL:
    # print(sql)

    with presto_engine.connect() as conn:
        sql_clean = sql.rstrip()
        if sql_clean.endswith(";"):
            sql_clean = sql_clean[:-1]

        df = pd.read_sql(sql_clean, conn)

    # Normalize types for downstream joins
    #if not df.empty:
   #     df["icp_client"] = df["icp_client"].astype(str)
    #    df["observe_scorecard"] = df["observe_scorecard"].str.lower()

    return df

In [7]:
df = query_expert_assist_presto_group(
        start_date,
        end_date,
        client_list,
        business_unit_list,
    )

In [8]:
df

,week_,BusinessUnit_,Client_,expert_id,num,den,calc
0,2026-01-09,S,A,685532,2,28,0.071
1,2026-01-16,S,V,676576,19,38,0.500
2,2026-01-09,M,A,688527,0,105,0.000
3,2026-01-02,M,V,699964,0,99,0.000
4,2026-01-09,S,V,691973,26,60,0.433
...,...,...,...,...,...,...,...
37411,2026-01-30,S,A,701686,10,13,0.769
37412,2026-01-23,M,A,689347,0,1,0.000
37413,2025-12-19,M,A,661957,0,25,0.000
37414,2026-01-30,M,V,124839,0,5,0.000


In [9]:
# Save to CSV in the same directory

from pathlib import Path

file = Path("ea_output.csv")   # replace with your filename

if file.exists():
    file.unlink()
    df.to_csv("ea_output.csv", index=False)
else:
    df.to_csv("ea_output.csv", index=False)